In [ ]:
from datetime import datetime, UTC
from pathlib import Path

from scitacean import Thumbnail
from scitacean.model import DownloadDataset, DownloadTechnique
from scitacean.testing.client import FakeClient
from scitacean.testing.transfer import FakeFileTransfer

from scicat_widget import DatasetUploadWidget

transfer = FakeFileTransfer()
client = FakeClient.without_login("staging.ess", file_transfer=transfer)

w = DatasetUploadWidget(client, skip_confirm=False)
w

In [ ]:
assert len(client.datasets) == 1

In [ ]:
dataset = next(iter(client.datasets.values()))
unpredictable_fields = {
    name: getattr(dataset, name)
    for name in [
        "creationTime",
        "createdAt",
        "createdBy",
        "updatedAt",
        "updatedBy",
        "pid"
    ]
}

expected = DownloadDataset(
    accessGroups=["A1", "tillgång"],
    contactEmail="skinner@princi.pal",
    creationLocation="TEST:widget",
    datasetName="Test dataset name",
    description="Test dataset description\nSecond line.",
    endTime=datetime(1985, 2, 11, 16, 0, 13).astimezone(UTC),
    inputDatasets=[
        "abc/1234.5bd6",
        "76.feda",
    ],
    instrumentIds=["I/DIF2"],
    keywords=[],
    numberOfFiles=1,
    numberOfFilesArchived=0,
    orcidOfOwner="https://orcid.org/0000-0000-0000-0001;;",
    owner="Billy Ownersson;;Martha the Wise",
    ownerEmail="b@ownersson.com;stibbons@uu.am;",
    ownerGroup="135246",
    packedSize=0,
    principalInvestigators=["Principal Skinner"],
    proposalIds=["dada.663", "0098.aa.2"],
    sampleIds=["Probe Eins"],
    size=18,
    sourceFolder="/source/folder",
    startTime=datetime(1984, 7, 24).astimezone(UTC),
    techniques=[
        DownloadTechnique(name="photon probe", pid="http://purl.org/pan-science/PaNET/PaNET00100"),
    ],
    type="derived",
    usedSoftware=[
        "scitacean==26.6.0",
        "Thingamatronic = 2.0",
    ],
    **unpredictable_fields,
)
for key in DownloadDataset.model_fields.keys():
    actual = getattr(dataset, key)
    exp = getattr(expected, key)
    assert actual == exp, f"{key}: {actual} != {exp}"

In [ ]:
assert len(transfer.files) == 1

In [ ]:
expected_file_content = Path("data.csv").read_bytes()
assert transfer.files["/source/folder/data.csv"] == expected_file_content

In [ ]:
assert len(client.attachments) == 1

In [ ]:
[attachment] = client.attachments[dataset.pid]
assert attachment.caption == "Pretty picture"
assert (
        attachment.thumbnail.encoded_data()
        == Thumbnail.load_file("scicat.png").encoded_data()
)